# ME 280 HW 1

I am starting to like these workbooks. Though, I would still like it to be 100% markdown with runnable block-code cells. Anyway, before I do anything, I wanted to create an enum for all cards. According to [the official docs](https://docs.python.org/3/library/enum.html), there's no enum keyword in this language but there is an abstract class to extend from.


In [ ]:
from enum import Enum


# Enums are just class inherence here, nothing special here.
# I personally like CamelCase for the entries but it's acceptable
# in my eyes to just follow the docs and do SCREAMING_SNAKE_CASE.
# Oh also, the numbers here are arbitrary, they're not the actual values.
class Card(Enum):

    # I gave them discriminators somewhat matching with their values
    # so that makes it a 1-based indexing solution. I will handle the
    # worth of aces and face cards in a separate method.
    ACE = 1
    TWO = 2
    THREE = 3
    FOUR = 4
    FIVE = 5
    SIX = 6
    SEVEN = 7
    EIGHT = 8
    NINE = 9
    TEN = 10
    JACK = 11
    QUEEN = 12
    KING = 13

    # Declaring a worth method here since the indices are arbitrary. Furthermore,
    # I have also added the treat_ace_as_1 parameter to toggle between the
    # two possible ways of treating an ace as 1 or 11. My most used language is
    # TypeScript so I will not be hesitating from using type annotations.
    def worth(self, treat_ace_as_1=True) -> int:

        # Handling the ace is simple.
        if self == Card.ACE:

            # I am not sure if I like Python's ternary operator syntax but
            # I guess I am glad it even exists in the first place.
            return 1 if treat_ace_as_1 else 11

        # This solution to checking if the card falls in this list of face cards
        # is clean but I don't seem to be able to move the list into a static
        # field. This is definitely a performance hit.
        elif self in [Card.JACK, Card.QUEEN, Card.KING]:
            return 10

        # The rest of the cards have the correct values in their
        # enum discriminators.
        else:
            return self.value

    # During debugging, I found it useful to use the card emojis to
    # view the deck. It remains unused for the final submission.
    def __str__(self):

        # Switch cases for the win! Emojis from
        # https://www.piliapp.com/emoji/list/playing-cards/
        case = {
            Card.ACE: "🂱",
            Card.TWO: "🂲",
            Card.THREE: "🂳",
            Card.FOUR: "🂴",
            Card.FIVE: "🂵",
            Card.SIX: "🂶",
            Card.SEVEN: "🂷",
            Card.EIGHT: "🂸",
            Card.NINE: "🂹",
            Card.TEN: "🂺",
            Card.JACK: "🂻",
            Card.QUEEN: "🂽",
            Card.KING: "🂾",
        }

        return case[self]

It would be a great idea to test all these methods and variants out.


In [ ]:
print(Card.ACE, Card.ACE.worth())
print(Card.ACE, Card.ACE.worth(False))
print(Card.TWO, Card.TWO.worth())
print(Card.THREE, Card.THREE.worth())
print(Card.FOUR, Card.FOUR.worth())
print(Card.FIVE, Card.FIVE.worth())
print(Card.SIX, Card.SIX.worth())
print(Card.SEVEN, Card.SEVEN.worth())
print(Card.EIGHT, Card.EIGHT.worth())
print(Card.NINE, Card.NINE.worth())
print(Card.TEN, Card.TEN.worth())
print(Card.JACK, Card.JACK.worth())
print(Card.QUEEN, Card.QUEEN.worth())
print(Card.KING, Card.KING.worth())

🂱 1
🂱 11
🂲 2
🂳 3
🂴 4
🂵 5
🂶 6
🂷 7
🂸 8
🂹 9
🂺 10
🂻 10
🂽 10
🂾 10


Another every important thing I would like to implement is all the different sorts of player behavior my "game engine" can support. Initially I started off with just the dealer and the player but I realized there are a lot of funny names and behaviors I can come up with. Here's the 4 that I have at the moment:

- Drunk Gambler: this guy does a coin flip on either to hit or stand.
- High School Statistician: this person actually paid attention to statistics in high school and aims to stand just before `21 - average - 1` so that it's unlikely to bust if he were to go again. The `average` here is the average worth of all available cards.
- Seasoned Rich Guy: this guy looked up how to play Black Jack once on Google and has been using the same strategy ever since. Doesn't matter if he wins or now, he's rich so he doesn't care and won't change.
- Dealer: the dealer follows the standard <17 hit rule. Because of how I have this set up, you can make the dealer play against another dealer if you really wanted to. Actually, this is exactly what I do in the CSV file.


In [ ]:
class PlayerBehavior(Enum):

    # Naming enums became real fun here.
    DRUNK_GAMBLER = 0
    HIGH_SCHOOL_STATISTICIAN = 1
    SEASONED_RICH_GUY = 2
    DEALER = 3

    # String representation will be useful when generating the CSV. The
    # SCREAMING_SNAKE_CASE is nice for coding but awful for sheets.
    def __str__(self):
        case = {
            PlayerBehavior.DRUNK_GAMBLER: "Drunk Gambler",
            PlayerBehavior.HIGH_SCHOOL_STATISTICIAN: "High School Statistician",
            PlayerBehavior.SEASONED_RICH_GUY: "Seasoned Rich Guy",
            PlayerBehavior.DEALER: "Dealer",
        }

        return case[self]

In [ ]:
# Dear beloved random library, I welcome you to yet another project of mine.
import random


class Player:
    def __init__(
        self,
        behavior: PlayerBehavior,
        initial_cards: int,
        treat_ace_as_1: bool,
    ):
        self.behavior = behavior
        self.standing = False

        self.cards: list[Card] = []
        self.hit(initial_cards)

        self.treat_ace_as_1 = treat_ace_as_1

    def title_string(self):
        return f"{self.name}: {self.worth()}"

    def deck_string(self):
        return " ".join(str(card) for card in self.cards)

    def draw_random():
        return random.choice(list(Card))

    def upcard(self):
        return self.cards[0]

    def hit(self, hits=1):
        for _ in range(hits):
            self.cards.append(Player.draw_random())

    def stand(self):
        self.standing = True

    def worth(self):
        return sum(card.worth(self.treat_ace_as_1) for card in self.cards)

    def is_bust(self):
        return self.worth() > 21

    def decide(self, opponent: "Player"):
        if self.behavior == PlayerBehavior.DRUNK_GAMBLER:
            if random.random() < 0.5:
                self.hit()
            else:
                self.stand()

        elif self.behavior == PlayerBehavior.HIGH_SCHOOL_STATISTICIAN:
            values = [card.worth(self.treat_ace_as_1) for card in Card]
            average = sum(values) / len(values)

            if self.worth() < 21 - average - 1:
                self.hit()
            else:
                self.stand()

        elif self.behavior == PlayerBehavior.SEASONED_RICH_GUY:
            dealer_card = opponent.upcard().worth()

            total = self.worth()

            if self.treat_ace_as_1:
                if total <= 11:
                    self.hit()
                elif 12 <= total <= 16 and dealer_card >= 7:
                    self.hit()
                else:
                    self.stand()
            else:
                if total <= 17:
                    self.hit()
                elif total == 18 and dealer_card in [9, 10, 11]:
                    self.hit()
                else:
                    self.stand()

        elif self.behavior == PlayerBehavior.DEALER:
            if self.worth() < 17:
                self.hit()
            else:
                self.stand()

    def play(self, opponent: "Player"):
        while not self.standing and not self.is_bust():
            self.decide(opponent)

In [301]:
class GameResult(Enum):
    PLAYER_WINNER = 0
    DEALER_WINNER = 1
    DRAW = 2

    def worth(self):
        case = {
            GameResult.PLAYER_WINNER: 1,
            GameResult.DEALER_WINNER: -1,
            GameResult.DRAW: 0,
        }

        return case[self]

In [302]:
class BlackJack:
    def __init__(
        self, player_behavior=PlayerBehavior.DRUNK_GAMBLER, treat_ace_as_1=True
    ):
        self.player = Player(
            player_behavior,
            2,
            treat_ace_as_1,
        )
        self.dealer = Player(
            PlayerBehavior.DEALER,
            1,
            treat_ace_as_1,
        )

    def play(self):
        self.player.play(self.dealer)

        if self.player.is_bust():
            return GameResult.DEALER_WINNER

        self.dealer.play(self.player)

        if self.dealer.is_bust() or self.player.worth() > self.dealer.worth():
            return GameResult.PLAYER_WINNER

        if self.player.worth() < self.dealer.worth():
            return GameResult.DEALER_WINNER

        return GameResult.DRAW

In [303]:
class GameMatrix:
    def __init__(self, rounds: int):
        self.rounds = rounds

    def csv(self):
        draft = "Round, "
        draft += ", ".join(
            f"{player_behavior} ({"Ace=1" if treat_ace_as_1 else "Ace=11"})"
            for player_behavior in PlayerBehavior
            for treat_ace_as_1 in [True, False]
        )
        draft += "\n"

        for i in range(0, self.rounds):
            index = i + 1

            draft += f"{index}, "

            for player_behavior in PlayerBehavior:
                for treat_ace_as_1 in [True, False]:
                    game = BlackJack(player_behavior, treat_ace_as_1)
                    result = game.play()

                    draft += f"{result.worth()}, "

            draft += "\n"

        return draft

In [304]:
text = GameMatrix(100).csv()

with open("test.games.csv", "w") as f:
    f.write(text)
    f.close()

https://docs.google.com/spreadsheets/d/1QLnzs7N4hXCWY2EpTaZq5YAZbh5USvadVlXYtIN-6OQ
